# Embed ML-paper chunks on Colab GPU

This notebook turns `chunks.json` into `embeddings.npy` using the **BGE** embedding
model on a **free Colab GPU**. The heavy compute happens here; your laptop only
assembles the final index afterwards.

**Steps:**
1. Runtime -> Change runtime type -> **T4 GPU** (free).
2. Run every cell top to bottom.
3. When prompted, upload your local `data/chunks.json`.
4. At the end, `embeddings.npy` downloads automatically -- put it in your local `data/` folder.


In [ ]:
# 1) Confirm we actually have a GPU (should print a GPU name, not 'cpu').
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY -- enable GPU in Runtime settings!')

In [ ]:
# 2) Install the embedding library (sentence-transformers). ~1 min.
!pip install -q sentence-transformers==3.2.1

In [ ]:
# 3) Upload your local data/chunks.json when the file picker appears.
from google.colab import files
uploaded = files.upload()   # choose chunks.json

import json
chunks = json.load(open('chunks.json', encoding='utf-8'))
texts = [c['text'] for c in chunks]
print(f'Loaded {len(texts):,} chunks to embed.')

In [ ]:
# 4) Embed on the GPU. Must match the LOCAL settings exactly:
#    same model + normalize_embeddings=True  (so vectors are compatible).
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cuda')
vectors = model.encode(
    texts,
    batch_size=256,            # big batches are fine on GPU
    normalize_embeddings=True,
    show_progress_bar=True,
)
vectors = np.asarray(vectors, dtype='float32')
print('Embeddings shape:', vectors.shape)   # expect (num_chunks, 384)

In [ ]:
# 5) Save and download embeddings.npy -> put it in your local data/ folder.
import numpy as np
np.save('embeddings.npy', vectors)
from google.colab import files
files.download('embeddings.npy')
print('Done! Move embeddings.npy into your project\'s data/ folder.')